# Free Gaussian wave packet entropy validation with Crank-Nicolson

In this notebook I want to connect the analytic entropy work to the numerical method. Up to this point, I already calculated the entropy for a free Gaussian wave packet by hand. The next thing I want to check is whether a Crank-Nicolson solver gives the same entropy curve.

This is important because before I use the code for a potential where I do not know the answer, I need to test it on a case where I do know the answer.

The free particle Hamiltonian is

$$
\hat H
=
\frac{\hat p^2}{2m}
=
-\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2}.
$$

The time-dependent Schrodinger equation is

$$
i\hbar\frac{\partial \psi(x,t)}{\partial t}
=
\hat H\psi(x,t).
$$

The entropy I am tracking is the differential Shannon entropy

$$
S_x(t)
=
-\int_{-\infty}^{\infty}\rho(x,t)\ln\rho(x,t)\,dx,
$$

where

$$
\rho(x,t)=|\psi(x,t)|^2.
$$

The plan is simple:

1. start with a normalized Gaussian wave packet,
2. evolve it freely using Crank-Nicolson,
3. compute the numerical density at each time,
4. calculate norm, mean position, variance, and entropy,
5. compare the numerical entropy with the analytic expression.


## 1. Analytic result I am trying to reproduce

I use the initial Gaussian

$$
\psi(x,0)
=
\left(\frac{2\lambda}{\pi}\right)^{1/4}
\exp[-\lambda(x-x_0)^2]\exp(ik_0x).
$$

The phase factor $\exp(ik_0x)$ gives the packet an average momentum, but it does not change the initial density because

$$
|\exp(ik_0x)|^2=1.
$$

So the initial density is

$$
\rho(x,0)
=
\left(\frac{2\lambda}{\pi}\right)^{1/2}
\exp[-2\lambda(x-x_0)^2].
$$

For this density, the initial variance is

$$
\Delta x(0)^2=\frac{1}{4\lambda}.
$$

For a freely evolving Gaussian wave packet, the variance evolves as

$$
\Delta x(t)^2
=
\frac{1}{4\lambda}
\left[1+\left(\frac{2\hbar\lambda t}{m}\right)^2\right].
$$

I will write

$$
\alpha=\frac{2\hbar\lambda}{m}.
$$

Then

$$
\Delta x(t)^2
=
\frac{1}{4\lambda}(1+\alpha^2t^2).
$$

For a one-dimensional Gaussian density, the entropy is

$$
S_x(t)
=
\frac12\ln\left[2\pi e\Delta x(t)^2\right].
$$

Substituting the free-particle variance gives

$$
S_x(t)
=
\frac12\ln\left[
\frac{\pi e}{2\lambda}(1+\alpha^2t^2)
\right].
$$

This is the exact curve I want the numerical calculation to reproduce.


## 2. Numerical form of entropy

On a grid, the integral becomes a sum. If the grid spacing is $\Delta x$, then

$$
\int \rho(x,t)\,dx
\approx
\sum_j \rho_j(t)\Delta x.
$$

So the entropy becomes

$$
S_x(t)
\approx
-\sum_j \rho_j(t)\ln[\rho_j(t)]\Delta x.
$$

This is differential entropy, so the value depends on the length unit used for $x$. That is not a problem here because I am comparing the numerical entropy with the analytic entropy using the same dimensionless units.

One numerical detail is that very tiny values of $\rho$ can underflow close to zero. Since $\rho\ln\rho$ goes to zero as $\rho\to0$, I handle this by only using grid points where $\rho>0$ in the sum.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags, identity
from scipy.sparse.linalg import splu

%matplotlib inline


## 3. Choose units and physical parameters

For this validation test, I use dimensionless units:

$$
\hbar=1,
\qquad
m=1.
$$

I choose a Gaussian parameter $\lambda$ and then compute

$$
\alpha=\frac{2\hbar\lambda}{m}.
$$

The parameter $k_0$ controls the average momentum. I keep it available in the code because it is useful for later tests, but for the entropy comparison the main effect comes from the width, not the center motion.


In [ ]:
# Dimensionless constants
hbar = 1.0
m = 1.0

# Gaussian parameters
lam = 0.5
x0 = 0.0
k0 = 0.0

alpha = 2.0 * hbar * lam / m

print(f"hbar = {hbar}")
print(f"m = {m}")
print(f"lambda = {lam}")
print(f"alpha = {alpha}")


## 4. Build the spatial and time grids

I need a grid that is large enough so the packet does not reach the boundary during the run. If the boundary is too close, the wave packet can reflect from the edge, and then the entropy will no longer match the free-particle analytic result.

I use

$$
x_{\min}\leq x\leq x_{\max}.
$$

The grid spacing is

$$
\Delta x=\frac{x_{\max}-x_{\min}}{N-1}.
$$

For time, I use

$$
t_n=n\Delta t.
$$


In [ ]:
# Spatial grid
N = 1600
x_min = -80.0
x_max = 80.0
x = np.linspace(x_min, x_max, N)
dx = x[1] - x[0]

# Time grid
dt = 0.01
t_final = 10.0
num_steps = int(t_final / dt)
save_every = 10

print(f"N = {N}")
print(f"dx = {dx:.6f}")
print(f"dt = {dt}")
print(f"number of time steps = {num_steps}")
print(f"saving every {save_every} steps")


## 5. Define the free-particle potential

For the first validation test,

$$
V(x)=0.
$$

This is the cleanest case because the analytic entropy is already known.


In [ ]:
# Free-particle potential
V = np.zeros_like(x)

plt.figure(figsize=(8, 4))
plt.plot(x, V)
plt.xlabel("x")
plt.ylabel("V(x)")
plt.title("Free-particle potential")
plt.grid(True, alpha=0.3)
plt.show()


## 6. Discretize the Hamiltonian

The kinetic energy operator is

$$
-\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2}.
$$

I approximate the second derivative using the central difference formula:

$$
\frac{\partial^2\psi}{\partial x^2}\bigg|_{x=x_j}
\approx
\frac{\psi_{j+1}-2\psi_j+\psi_{j-1}}{(\Delta x)^2}.
$$

Substituting into the Hamiltonian gives

$$
(H\psi)_j
=
-\frac{\hbar^2}{2m}
\frac{\psi_{j+1}-2\psi_j+\psi_{j-1}}{(\Delta x)^2}
+V_j\psi_j.
$$

So the matrix is tridiagonal. This is useful because sparse matrix methods can solve it efficiently.


In [ ]:
def build_hamiltonian(x, V, hbar=1.0, m=1.0):
    """Build the finite-difference Hamiltonian for a 1D potential."""
    dx = x[1] - x[0]
    N = len(x)

    main_diag = hbar**2 / (m * dx**2) + V
    off_diag = -hbar**2 / (2.0 * m * dx**2) * np.ones(N - 1)

    H = diags(
        [off_diag, main_diag, off_diag],
        offsets=[-1, 0, 1],
        format="csc",
    )
    return H

H = build_hamiltonian(x, V, hbar=hbar, m=m)
print(f"Hamiltonian shape: {H.shape}")
print(f"Number of nonzero entries: {H.nnz}")


## 7. Crank-Nicolson update

The exact time evolution is

$$
\psi(t+\Delta t)=e^{-i\hat H\Delta t/\hbar}\psi(t).
$$

Crank-Nicolson replaces this with the symmetric update

$$
\left(I+\frac{i\Delta t}{2\hbar}H\right)\psi^{n+1}
=
\left(I-\frac{i\Delta t}{2\hbar}H\right)\psi^n.
$$

I define

$$
A=I+\frac{i\Delta t}{2\hbar}H,
$$

and

$$
B=I-\frac{i\Delta t}{2\hbar}H.
$$

Then each step is

$$
A\psi^{n+1}=B\psi^n.
$$

Since the Hamiltonian is time independent here, $A$ does not change with time. I can factorize $A$ once and reuse that factorization.


In [ ]:
I = identity(N, format="csc")

A = I + 1j * dt / (2.0 * hbar) * H
B = I - 1j * dt / (2.0 * hbar) * H

A_lu = splu(A)
print("Crank-Nicolson matrices built and A factorized.")


## 8. Initial Gaussian wave packet

The analytic Gaussian I am using is

$$
\psi(x,0)
=
\left(\frac{2\lambda}{\pi}\right)^{1/4}
\exp[-\lambda(x-x_0)^2]\exp(ik_0x).
$$

On the finite grid, I still normalize numerically because the grid is not literally infinite.

The numerical normalization condition is

$$
\sum_j |\psi_j|^2\Delta x=1.
$$


In [ ]:
def initial_gaussian(x, lam, x0=0.0, k0=0.0):
    """Create the initial Gaussian wave packet used in the analytic derivation."""
    prefactor = (2.0 * lam / np.pi) ** 0.25
    envelope = np.exp(-lam * (x - x0) ** 2)
    phase = np.exp(1j * k0 * x)
    return prefactor * envelope * phase

def normalize(psi, dx):
    norm = np.sqrt(np.sum(np.abs(psi) ** 2) * dx)
    return psi / norm

psi = initial_gaussian(x, lam=lam, x0=x0, k0=k0)
psi = normalize(psi, dx)
psi_initial = psi.copy()

initial_norm = np.sum(np.abs(psi) ** 2) * dx
print(f"Initial norm = {initial_norm:.12f}")


## 9. Diagnostic functions

Before trusting entropy, I want to track basic quantities:

$$
\text{norm}(t)=\int \rho(x,t)\,dx,
$$

$$
\langle x\rangle(t)=\int x\rho(x,t)\,dx,
$$

$$
\Delta x(t)^2
=
\int (x-\langle x\rangle)^2\rho(x,t)\,dx,
$$

and

$$
S_x(t)=-\int \rho(x,t)\ln\rho(x,t)\,dx.
$$

If the norm drifts badly, then the entropy result is not reliable. If the variance does not match the analytic variance, then the entropy probably will not match either.


In [ ]:
def density(psi):
    return np.abs(psi) ** 2

def norm_of_density(rho, dx):
    return np.sum(rho) * dx

def mean_x(x, rho, dx):
    return np.sum(x * rho) * dx

def variance_x(x, rho, dx):
    mx = mean_x(x, rho, dx)
    return np.sum((x - mx) ** 2 * rho) * dx

def entropy_x(rho, dx):
    mask = rho > 0
    return -np.sum(rho[mask] * np.log(rho[mask])) * dx

def analytic_variance(t, lam, hbar=1.0, m=1.0):
    alpha = 2.0 * hbar * lam / m
    return (1.0 / (4.0 * lam)) * (1.0 + alpha**2 * t**2)

def analytic_entropy(t, lam, hbar=1.0, m=1.0):
    var = analytic_variance(t, lam, hbar=hbar, m=m)
    return 0.5 * np.log(2.0 * np.pi * np.e * var)

def analytic_mean(t, x0=0.0, k0=0.0, hbar=1.0, m=1.0):
    p0 = hbar * k0
    return x0 + (p0 / m) * t


## 10. Check the initial state against the analytic formulas

At $t=0$, the analytic variance is

$$
\Delta x(0)^2=\frac{1}{4\lambda}.
$$

The analytic entropy is

$$
S_x(0)=\frac12\ln\left(\frac{\pi e}{2\lambda}\right).
$$

I check these before starting the time evolution.


In [ ]:
rho0 = density(psi)

num_norm0 = norm_of_density(rho0, dx)
num_mean0 = mean_x(x, rho0, dx)
num_var0 = variance_x(x, rho0, dx)
num_entropy0 = entropy_x(rho0, dx)

ana_mean0 = analytic_mean(0.0, x0=x0, k0=k0, hbar=hbar, m=m)
ana_var0 = analytic_variance(0.0, lam=lam, hbar=hbar, m=m)
ana_entropy0 = analytic_entropy(0.0, lam=lam, hbar=hbar, m=m)

print(f"Numerical norm at t=0     = {num_norm0:.12f}")
print(f"Numerical mean at t=0     = {num_mean0:.12f}")
print(f"Analytic mean at t=0      = {ana_mean0:.12f}")
print(f"Numerical variance at t=0 = {num_var0:.12f}")
print(f"Analytic variance at t=0  = {ana_var0:.12f}")
print(f"Numerical entropy at t=0  = {num_entropy0:.12f}")
print(f"Analytic entropy at t=0   = {ana_entropy0:.12f}")


## 11. Plot the initial density

This is just a check that the initial state looks like the Gaussian I expect.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(x, rho0, label=r"$\rho(x,0)=|\psi(x,0)|^2$")
plt.xlabel("x")
plt.ylabel(r"$\rho(x,0)$")
plt.title("Initial free Gaussian density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 12. Evolve the wave packet

Now I evolve the wavefunction using the Crank-Nicolson update. At saved time steps, I store the density and the diagnostics.

The quantities I store are:

$$
\text{norm}(t),\quad \langle x\rangle(t),\quad \Delta x(t)^2,\quad S_x(t).
$$


In [ ]:
t_values = []
norm_values = []
mean_values = []
variance_values = []
entropy_values = []
snapshots = []
snapshot_times = []

# Times where I want to keep full density snapshots for plotting.
times_to_snapshot = np.array([0.0, 2.0, 4.0, 6.0, 8.0, 10.0])
snapshot_tolerance = dt / 2.0

psi = psi_initial.copy()

for n in range(num_steps + 1):
    t = n * dt

    if n % save_every == 0:
        rho = density(psi)
        t_values.append(t)
        norm_values.append(norm_of_density(rho, dx))
        mean_values.append(mean_x(x, rho, dx))
        variance_values.append(variance_x(x, rho, dx))
        entropy_values.append(entropy_x(rho, dx))

    if np.any(np.abs(times_to_snapshot - t) < snapshot_tolerance):
        snapshots.append(density(psi))
        snapshot_times.append(t)

    if n < num_steps:
        rhs = B @ psi
        psi = A_lu.solve(rhs)

# Convert diagnostics to arrays.
t_values = np.array(t_values)
norm_values = np.array(norm_values)
mean_values = np.array(mean_values)
variance_values = np.array(variance_values)
entropy_values = np.array(entropy_values)

print("Evolution complete.")
print(f"Saved {len(t_values)} diagnostic time points.")
print(f"Saved {len(snapshots)} density snapshots.")


## 13. Compare density snapshots

For the free particle, the density should spread out as time increases. If $k_0$ is nonzero, the center should also move. In this run I used $k_0=0$, so I mainly expect spreading without center motion.


In [ ]:
plt.figure(figsize=(9, 5))

for rho_snap, t_snap in zip(snapshots, snapshot_times):
    plt.plot(x, rho_snap, label=fr"$t={t_snap:.1f}$")

plt.xlabel("x")
plt.ylabel(r"$\rho(x,t)$")
plt.title("Free Gaussian density spreading in time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.xlim(-30, 30)
plt.show()


## 14. Norm check

Crank-Nicolson should preserve probability very well because the update is unitary for a Hermitian Hamiltonian. Numerically, I check that

$$
\sum_j |\psi_j(t)|^2\Delta x\approx 1.
$$


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(t_values, norm_values)
plt.xlabel("time")
plt.ylabel("norm")
plt.title("Norm conservation check")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Minimum norm = {norm_values.min():.12f}")
print(f"Maximum norm = {norm_values.max():.12f}")
print(f"Final norm   = {norm_values[-1]:.12f}")


## 15. Mean position check

The analytic mean position for a free packet is

$$
\langle x\rangle(t)=x_0+\frac{\hbar k_0}{m}t.
$$

For this run, $k_0=0$, so the mean should stay close to zero.


In [ ]:
mean_analytic = analytic_mean(t_values, x0=x0, k0=k0, hbar=hbar, m=m)

plt.figure(figsize=(8, 5))
plt.plot(t_values, mean_values, label="numerical")
plt.plot(t_values, mean_analytic, "--", label="analytic")
plt.xlabel("time")
plt.ylabel(r"$\langle x\rangle(t)$")
plt.title("Mean position validation")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

mean_error = np.max(np.abs(mean_values - mean_analytic))
print(f"Maximum mean-position error = {mean_error:.6e}")


## 16. Variance check

The analytic variance is

$$
\Delta x(t)^2
=
\frac{1}{4\lambda}(1+\alpha^2t^2).
$$

This is the most important check before comparing entropy, because for a Gaussian the entropy is controlled by the width.


In [ ]:
variance_analytic = analytic_variance(t_values, lam=lam, hbar=hbar, m=m)

plt.figure(figsize=(8, 5))
plt.plot(t_values, variance_values, label="numerical")
plt.plot(t_values, variance_analytic, "--", label="analytic")
plt.xlabel("time")
plt.ylabel(r"$\Delta x(t)^2$")
plt.title("Variance validation for the free Gaussian")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

variance_error = np.max(np.abs(variance_values - variance_analytic))
print(f"Maximum variance error = {variance_error:.6e}")


## 17. Entropy validation

Now I compare the numerical entropy

$$
S_x(t)
\approx
-\sum_j \rho_j(t)\ln\rho_j(t)\Delta x
$$

with the analytic expression

$$
S_x(t)
=
\frac12\ln\left[
\frac{\pi e}{2\lambda}(1+\alpha^2t^2)
\right].
$$

If these match, then the entropy calculation is working for the case where I already know the exact answer.


In [ ]:
entropy_analytic_values = analytic_entropy(t_values, lam=lam, hbar=hbar, m=m)

plt.figure(figsize=(8, 5))
plt.plot(t_values, entropy_values, label="numerical")
plt.plot(t_values, entropy_analytic_values, "--", label="analytic")
plt.xlabel("time")
plt.ylabel(r"$S_x(t)$")
plt.title("Entropy validation for the free Gaussian")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

entropy_error = entropy_values - entropy_analytic_values
print(f"Maximum absolute entropy error = {np.max(np.abs(entropy_error)):.6e}")
print(f"Initial numerical entropy = {entropy_values[0]:.12f}")
print(f"Initial analytic entropy  = {entropy_analytic_values[0]:.12f}")
print(f"Final numerical entropy   = {entropy_values[-1]:.12f}")
print(f"Final analytic entropy    = {entropy_analytic_values[-1]:.12f}")


## 18. Entropy error plot

This plot is useful because it shows where the numerical result starts to drift away from the analytic result. If the domain is too small, the error usually becomes worse when the packet reaches the boundaries. If the time step or grid spacing is too large, the error can appear earlier.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(t_values, entropy_error)
plt.xlabel("time")
plt.ylabel(r"$S_{\mathrm{num}}(t)-S_{\mathrm{analytic}}(t)$")
plt.title("Entropy error")
plt.grid(True, alpha=0.3)
plt.show()


## 19. Entropy derivative check

From the analytic entropy formula,

$$
S_x(t)=S_x(0)+\frac12\ln(1+\alpha^2t^2),
$$

so

$$
\frac{dS_x}{dt}
=
\frac{\alpha^2t}{1+\alpha^2t^2}.
$$

This connects to the continuity-equation idea. For the free Gaussian, the packet expands, so the entropy derivative should be positive for $t>0$.


In [ ]:
# Numerical derivative from saved entropy values.
dSdt_numeric = np.gradient(entropy_values, t_values)
dSdt_analytic = (alpha**2 * t_values) / (1.0 + alpha**2 * t_values**2)

plt.figure(figsize=(8, 5))
plt.plot(t_values, dSdt_numeric, label="numerical derivative")
plt.plot(t_values, dSdt_analytic, "--", label="analytic derivative")
plt.xlabel("time")
plt.ylabel(r"$dS_x/dt$")
plt.title("Entropy rate for the free Gaussian")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 20. What I learned from this validation

The main point of this notebook is not just to make a plot. The point is to check that the numerical pipeline is trustworthy.

The pipeline is

$$
\psi(x,t)
\rightarrow
\rho(x,t)=|\psi(x,t)|^2
\rightarrow
\text{norm},\ \langle x\rangle,\ \Delta x^2,\ S_x(t).
$$

For the free Gaussian, the numerical result should reproduce three things:

1. the norm stays close to 1,
2. the variance grows like $\frac{1}{4\lambda}(1+\alpha^2t^2)$,
3. the entropy grows like $\frac12\ln\left[\frac{\pi e}{2\lambda}(1+\alpha^2t^2)\right]$.

Once this works, I can use the same code structure for the harmonic oscillator and then later for the Morse potential. The free-particle case is the calibration step. If I cannot reproduce this analytic result, then it would not make sense to trust the entropy calculation for a potential where I do not know the exact answer.


## 21. Next step after this notebook

The next notebook should repeat the same validation idea for the harmonic oscillator potential,

$$
V(x)=\frac12m\omega^2x^2.
$$

For the translated harmonic oscillator ground state, I expect

$$
\langle x\rangle(t)=\xi\cos(\omega t),
$$

$$
\Delta x^2(t)=\frac{\hbar}{2m\omega},
$$

and

$$
S_x(t)=\frac12\ln\left(\frac{\pi e\hbar}{m\omega}\right).
$$

That will test a different idea: the state is time dependent, but the entropy stays constant because the packet does not spread.
